# 01_data_check

This notebook checks the basic status of the dataset `data/raw/YRBS_2007.csv`.

Workflow:
1. Load the raw data
2. Show `df.describe()`
3. Check missing values by rows and by columns
4. Show value-distribution ratios for each column
5. Present continuous and discrete variables separately


In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f"{x:,.4f}")

In [ ]:
# Read raw data
df = pd.read_csv("data/raw/YRBS_2007.csv")

print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

## 1. Basic summary with `df.describe()`

In [ ]:
# Basic descriptive statistics for numeric columns
df.describe().T

## 2. Missing-value check by row and by column

In [ ]:
# Missing values by row
row_missing_count = df.isna().sum(axis=1)
row_missing_ratio = df.isna().mean(axis=1)

row_missing_summary = pd.DataFrame({
    "missing_count": row_missing_count,
    "missing_ratio": row_missing_ratio
})

print("Top 10 rows with the most missing values:")
display(row_missing_summary.sort_values(["missing_count", "missing_ratio"], ascending=False).head(10))

print("\nDistribution of missing counts across rows:")
display(row_missing_count.value_counts().sort_index().rename_axis("missing_count").reset_index(name="row_frequency"))

In [ ]:
# Missing values by column
col_missing_count = df.isna().sum(axis=0)
col_missing_ratio = df.isna().mean(axis=0)

column_missing_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": col_missing_count.values,
    "missing_ratio": col_missing_ratio.values,
    "non_missing_count": df.notna().sum(axis=0).values,
    "unique_non_missing": df.nunique(dropna=True).values
}).sort_values(["missing_count", "missing_ratio", "unique_non_missing"], ascending=[False, False, True])

column_missing_summary

## 3. Separate continuous and discrete variables

In [ ]:
# Heuristic split:
# - object/category/bool -> discrete
# - numeric with small number of distinct values (<= 15) -> discrete
# - otherwise -> continuous
#
# You can adjust the threshold if your instructor wants a different rule.

DISCRETE_UNIQUE_THRESHOLD = 15

discrete_cols = []
continuous_cols = []

for col in df.columns:
    series = df[col]
    non_na = series.dropna()
    nunique = non_na.nunique()

    if pd.api.types.is_object_dtype(series) or pd.api.types.is_bool_dtype(series):
        discrete_cols.append(col)
    elif pd.api.types.is_numeric_dtype(series):
        if nunique <= DISCRETE_UNIQUE_THRESHOLD:
            discrete_cols.append(col)
        else:
            continuous_cols.append(col)
    else:
        discrete_cols.append(col)

print(f"Discrete columns: {len(discrete_cols)}")
print(f"Continuous columns: {len(continuous_cols)}")

display(pd.DataFrame({
    "discrete_columns": pd.Series(discrete_cols),
    "continuous_columns": pd.Series(continuous_cols)
}))

## 4. Value-distribution ratios for discrete variables

In [ ]:
def discrete_distribution_table(dataframe, columns):
    tables = []
    for col in columns:
        s = dataframe[col]
        freq = s.value_counts(dropna=False).sort_index(key=lambda x: x.map(str))
        ratio = s.value_counts(dropna=False, normalize=True).sort_index(key=lambda x: x.map(str))
        temp = pd.DataFrame({
            "column": col,
            "value": freq.index.astype(str),
            "count": freq.values,
            "ratio": ratio.values
        })
        tables.append(temp)
    return pd.concat(tables, ignore_index=True)

discrete_distribution = discrete_distribution_table(df, discrete_cols)
discrete_distribution

## 5. Distribution summary for continuous variables

In [ ]:
continuous_distribution_summary = pd.DataFrame({
    "column": continuous_cols,
    "non_missing_count": [df[col].notna().sum() for col in continuous_cols],
    "missing_count": [df[col].isna().sum() for col in continuous_cols],
    "missing_ratio": [df[col].isna().mean() for col in continuous_cols],
    "unique_non_missing": [df[col].nunique(dropna=True) for col in continuous_cols],
    "min": [df[col].min() for col in continuous_cols],
    "q1": [df[col].quantile(0.25) for col in continuous_cols],
    "median": [df[col].median() for col in continuous_cols],
    "mean": [df[col].mean() for col in continuous_cols],
    "q3": [df[col].quantile(0.75) for col in continuous_cols],
    "max": [df[col].max() for col in continuous_cols],
    "std": [df[col].std() for col in continuous_cols]
}).sort_values("column").reset_index(drop=True)

continuous_distribution_summary

## 6. Optional quick check for one column

In [ ]:
# Example: replace the column name below if you want to inspect one variable in detail
example_col = "CurrentAlcoholUse"

if example_col in discrete_cols:
    detail = pd.DataFrame({
        "count": df[example_col].value_counts(dropna=False),
        "ratio": df[example_col].value_counts(dropna=False, normalize=True)
    })
    display(detail)
elif example_col in continuous_cols:
    display(df[example_col].describe())
else:
    print(f"{example_col} was not found in the automatic split.")

## Notes

- This notebook is for **data checking** and **distribution inspection**.
- For the project, you can later move selected variables into:
  - `02_eda.ipynb`
  - `03_inference.ipynb`
- If your instructor wants a stricter rule for continuous vs. discrete variables, edit `DISCRETE_UNIQUE_THRESHOLD`.
